In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd drive/MyDrive/UNISINOS/IA\ Aplicada\ à\ Saúde

/content/drive/MyDrive/UNISINOS/IA Aplicada à Saúde


In [ ]:
!ls

'Tutorial 01.ipynb'  'Tutorial 02.ipynb'


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import confusion_matrix, classification_report

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
batch_size = 32
num_epochs = 5
learning_rate = 0.001

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Dispositivo em uso {device}')

Dispositivo em uso cpu


In [ ]:
transforms = transforms.Compose([
    transforms.Resize([128, 128]),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
    ])

In [ ]:
train_data = datasets.ImageFolder('../chest_xray/chest_xray/train', transform=transforms)
val_data = datasets.ImageFolder('../chest_xray/chest_xray/val', transform=transforms)
test_data = datasets.ImageFolder('../chest_xray/chest_xray/test', transform=transforms)

In [ ]:
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
class_names = train_data.classes

print(f'As classes do dataset são: {class_names}')

As classes do dataset são: ['NORMAL', 'PNEUMONIA']


In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(2)

    self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 64 * 64, 128),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128, 2)
    )

  def forward(self, x):
    x = nn.ReLU()(self.conv1(x))
    x = self.pool(nn.ReLU()(self.conv2(x)))

    x = self.fc(x)

    return x

In [ ]:
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
## treinamento

for epoch in range(num_epochs):
  model.train()

  train_loss = 0.0
  correct = 0

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item() * images.size(0)

    correct += (outputs.argmax(1) == labels).sum().item()

  train_loss /= len(train_loader.dataset)
  train_accuracy = correct / len(train_loader.dataset)

  model.eval()
  val_correct = 0

  with torch.no_grad():
    for images, labels in val_loader:
      images, labels = images.to(device), labels.to(device)

      preds = model(images)
      val_correct += (preds.argmax(1) == labels).sum().item()
    val_acc = val_correct / len(val_data)

  print(f'Época {epoch+1}/{num_epochs} - Loss {train_loss:.4f}')
  print(f'Train Acc {train_accuracy:.4f} - Val Acc {val_acc:.4f}')

Época 1/5 - Loss 0.3296
Train Acc 0.9066 - Val Acc 0.8750
Época 2/5 - Loss 0.1220
Train Acc 0.9565 - Val Acc 0.5625
Época 3/5 - Loss 0.0875
Train Acc 0.9697 - Val Acc 0.5625
Época 4/5 - Loss 0.0721
Train Acc 0.9747 - Val Acc 0.7500
Época 5/5 - Loss 0.0559
Train Acc 0.9795 - Val Acc 0.8750


In [ ]:
def evaluate_model(model, name):

  model.eval()
  all_preds = []
  all_labels = []

  correct = 0

  with torch.no_grad():
    for images, labels in test_loader:
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)

      preds = outputs.argmax(1)

      correct += (preds == labels).sum().item()

      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  acc = correct / len(test_data)

  print(f'{name} - Acurácia no Teste {acc:.end}')
  cm = confusion_matrix(all_labels, all_preds)
  print('Relatório de Classificação:')

  print(classification_report(all_labels, all_preds, target_names=class_names))

  plt.figure(figsize=(6,6))

  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
  plt.xlabel('Predito')
  plt.ylabel('Verdadeiro')
  plt.title(f'Matriz de confusão - {name}')

  plt.show()
